# NAOMI encoder training (Colab)

Trains `nsm_ct.encoder_model.EncoderModel` (the candidate-lattice encoder, `dev/ENCODER_MODEL_SPEC.md`) on `runs/encoder_gold_v2.jsonl`, then reports:

1. English candidate-set recall (sense / slot / structure) vs a random-legal baseline, on a held-out split.
2. The Spanish grammar-swap eval on `runs/spanish_gold_v2.jsonl` -- the same English-trained weights evaluated on Spanish with zero Spanish training, vs a random baseline.

Run the three code cells below in order. Everything (repo clone, deps, USVS build, gold-data fetch, training, eval) happens inside this notebook -- no local setup needed.

In [ ]:
!git clone -b claude/m27-m28-cleanup https://github.com/LinguisticsDevelopment/NAOMI.git && cd NAOMI/consciousness_transformer && pip install -q torch numpy nltk pytest && pip install -e .

In [ ]:
%cd NAOMI/consciousness_transformer
import os
os.environ["NLTK_ALLOW_PROXIED_URLOPEN"] = "1"
!python -c "import nltk; nltk.download('wordnet', quiet=True); nltk.download('omw-1.4', quiet=True); nltk.download('omw-2.0', quiet=True)"
!python scripts/build_usvs.py
!mkdir -p runs
!git show origin/encoder-gold-v2:consciousness_transformer/runs/encoder_gold_v2.jsonl > runs/encoder_gold_v2.jsonl
!git show origin/spanish-gold-v2:consciousness_transformer/runs/spanish_gold_v2.jsonl > runs/spanish_gold_v2.jsonl
!wc -l runs/encoder_gold_v2.jsonl runs/spanish_gold_v2.jsonl

In [ ]:
!python scripts/colab_train_encoder.py --records 984 --epochs 50 --out runs/encoder_colab.pt

## Reading the RESULTS block

The last cell prints a `RESULTS` block with:

- **English (held-out test split):** `model` vs `random` candidate-set recall for sense / slot / structure -- the model row should clear the random-legal baseline by a wide margin if training worked.
- **Spanish grammar-swap:** the SAME English-trained checkpoint evaluated on `runs/spanish_gold_v2.jsonl`, again `model` vs `random`. This is the interlingua claim: one frozen encoder, a swapped-language front end, no Spanish training data.
- **policy params / MB, device, wall-clock** -- the model is small (sub-2 MB) and, as of this snapshot of `nsm_ct/encoder_model.py`, CPU-only: the driver detects that the model has no `.to(device)`/CUDA support (its controller builds plain CPU index tensors for every embedding lookup) and trains on CPU even if a GPU runtime is selected, printing a note to that effect. Selecting a GPU runtime in Colab will not speed this up until that's fixed upstream -- use a CPU runtime, no need to burn a GPU allocation.

`--records`/`--epochs` are tunable: `--records 984 --epochs 50` reproduces the spec's full-Stage-i split (788/98/98 of the 985 available English gold records) and is sized (from this repo's own CPU smoke timings) to land in roughly the 45-90 minute range end to end. Lower either for a faster, noisier run.

### Getting the checkpoint out

The trained checkpoint is saved to `runs/encoder_colab.pt` (relative to `NAOMI/consciousness_transformer`, i.e. `/content/NAOMI/consciousness_transformer/runs/encoder_colab.pt` in a fresh Colab runtime). Either:

```python
from google.colab import files
files.download('runs/encoder_colab.pt')
```

or mount Drive and copy it there before the runtime recycles:

```python
from google.colab import drive
drive.mount('/content/drive')
!cp runs/encoder_colab.pt /content/drive/MyDrive/encoder_colab.pt
```